# Lesson 05 — Orbits around a black hole

**Phase 4 of the gravsim project, part II: the exact Schwarzschild geodesics.**

The 1PN correction of Lesson 04 is the first term of an expansion in $GM/(c^2 r)$. Around
the Sun that parameter is $10^{-8}$ and the expansion is superb. Around a compact mass it can
approach $1/6$ — and then qualitatively new things happen: orbits that whirl many times at
nearly constant radius before escaping, a last stable circular orbit, a sphere where light itself
orbits, and capture. All of it follows from one function, the effective potential.

By the end you should be able to

1. write the Schwarzschild effective potential and read off circular orbits, the ISCO and the
   photon sphere;
2. show that the exact perihelion advance agrees with 1PN for Mercury to better than $0.01''$/cy,
   and quantify where 1PN fails;
3. integrate exact geodesics and produce precessing, zoom-whirl and plunging orbits;
4. distinguish proper time from coordinate time along an orbit.

## 1. The physics: one function for everything

Outside a spherical mass the metric is Schwarzschild's. With the gravitational radius
$\mu = GM/c^2$ and the Schwarzschild radius $r_s = 2\mu$, in the plane of the orbit,

$$ c^2\,d\tau^2 = \Big(1-\frac{r_s}{r}\Big)c^2dt^2 - \frac{dr^2}{1-r_s/r} - r^2 d\phi^2 . $$

Two symmetries (time translation, rotation) give two constants per unit mass,

$$ E = \Big(1-\frac{r_s}{r}\Big)c^2\frac{dt}{d\tau}, \qquad L = r^2\frac{d\phi}{d\tau}, $$

and the normalisation of the four-velocity turns the radial motion into a one-dimensional
problem in proper time:

$$ \tfrac12\Big(\frac{dr}{d\tau}\Big)^2 + V_{\rm eff}(r) = \frac{E^2/c^2 - c^2}{2}, \qquad
V_{\rm eff}(r) = -\frac{GM}{r} + \frac{L^2}{2r^2} - \frac{GM L^2}{c^2 r^3}. $$

The first two terms are Newton's; the third is the *only* relativistic term, and at small $r$
it overturns the centrifugal barrier. Consequences (`gravsim.metric.schwarzschild`):

* **circular orbits** at $dV_{\rm eff}/dr = 0$: $r_\pm = \frac{L^2}{2GM}\big[1\pm\sqrt{1-12\mu^2c^2/L^2}\,\big]$,
  the outer stable, the inner unstable;
* they merge at the **ISCO** $r = 6\mu$ when $L^2 = 12\,\mu^2c^2$; for smaller $L$ there is no
  circular orbit and every inward-moving particle is captured;
* the unstable branch ends at the **photon sphere** $r = 3\mu$ as $L\to\infty$;
* **shape of the orbit** (Binet, $u = 1/r$, $' = d/d\phi$): $u'' + u = GM/L^2 + 3\mu\,u^2$; dropping
  the last term gives Kepler's conic. Perturbation theory in $\mu u$ gives
  $\Delta\phi = 6\pi\mu/p$ per orbit (the 1PN result) with corrections of relative order $7.5\,\mu/p$;
* **time dilation**: $dt/d\tau = E/\big(c^2(1-r_s/r)\big) > 1$.

**Units.** gravsim keeps $GM$ and $c$ explicit. In this lesson we use two settings: the real
Sun ($GM_\odot$, $c = 173.14$ AU/day, $\mu = 1.48$ km) for Mercury, and *geometric units*
$GM = c = 1$ (so $\mu = 1$, $r_s = 2$, ISCO $= 6$) for the strong field. In the second setting a
radius of 10 means ten gravitational radii, whatever the mass.

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

from gravsim import constants as C
from gravsim.analysis import state_to_elements
from gravsim.data.horizons import load_j2000
from gravsim.metric import schwarzschild as sch
from gravsim.viz import PALETTE, figure_grid, plot_effective_potential

plt.rcParams["figure.dpi"] = 100

## 2. Scales: how far is Mercury from the strong field?

In [ ]:
mu_sun = C.GM_SUN / C.C_AU_DAY ** 2
two = load_j2000().subset(("Sun", "Mercury")).to_barycentric()
r, v = two.relative_to("Sun")
el = state_to_elements(r[1], v[1], float(two.gm.sum()))
a, e, period = float(el["a"]), float(el["e"]), float(el["period"])
p = a * (1 - e * e)
print(f"gravitational radius of the Sun  mu = GM/c^2 = {mu_sun:.3e} AU = {mu_sun * C.AU_KM:.3f} km")
print(f"Schwarzschild radius             r_s = 2 mu  = {sch.schwarzschild_radius(C.GM_SUN) * C.AU_KM:.3f} km")
print(f"photon sphere 3 mu = {sch.photon_sphere_radius(C.GM_SUN) * C.AU_KM:.2f} km,  ISCO 6 mu = {sch.isco_radius(C.GM_SUN) * C.AU_KM:.2f} km")
print(f"Mercury: p = a(1-e^2) = {p:.4f} AU = {p / mu_sun:.2e} mu  ->  mu/p = {mu_sun / p:.2e}")
print(f"1PN advance 6 pi mu/p = {6 * math.pi * mu_sun / p * C.ARCSEC_PER_RAD:.4f} arcsec/orbit; "
      f"next order ~ 7.5 mu/p = {7.5 * mu_sun / p:.1e} relative")

## 3. The effective potential explorer

In geometric units ($GM = c = 1$) the potential depends only on $L$ (in units of $\mu c$).
Below $L = \sqrt{12} = 3.46$ there is no well at all; at $L = 4$ a barrier at $r = 4$ with a well at
$r = 12$; for large $L$ the barrier moves toward the photon sphere at $r = 3$ and the well toward
Newton's $r = L^2$. A particle with "energy" $\mathcal E = (E^2/c^2 - c^2)/2$ above the barrier top
plunges; below it, and inside the well, it is bound and precesses; just below the top it
*whirls* near the unstable circular orbit before coming back out.

In [ ]:
gm, c = 1.0, 1.0
r_grid = np.linspace(2.05, 60.0, 2000)
fig, axes = figure_grid(4, ncols=2, panel=(5.2, 3.6))
for ax, L in zip(axes, (3.2, math.sqrt(12.0), 4.0, 5.0)):
    v_gr = sch.effective_potential(r_grid, L, gm, c)
    v_n = sch.newton_effective_potential(r_grid, L, gm)
    r_st, r_un = sch.circular_orbit_radii(L, gm, c)
    marks = {"photon sphere": 3.0, "ISCO": 6.0}
    if np.isfinite(r_st):
        marks[f"stable r = {r_st:.1f}"] = r_st
        marks[f"unstable r = {r_un:.1f}"] = r_un
    plot_effective_potential(r_grid, v_gr, v_n, ax=ax, marks=marks, xlabel="r [GM/c²]", ylabel="V_eff [c²]")
    ax.set_xlim(2.0, 40.0)
    ax.set_title(f"L = {L:.2f} GM/c")

## 4. Mercury: the exact orbit agrees with 1PN

`precession_per_orbit_numeric` integrates the Binet equation from Mercury's perihelion for
50 orbits and measures the mean advance between successive perihelia. Exact and 1PN agree to
$10^{-4}$ arcsec per century — the resolution of the numerical perihelion detection; the true
2PN difference, $7.5\,\mu/p \approx 2\times10^{-7}$ relative, is smaller still. (ROADMAP
target: $< 0.01''$/cy.)

In [ ]:
exact = sch.precession_per_orbit_numeric(a, e, C.GM_SUN, C.C_AU_DAY, n_orbits=50)
pn1 = sch.precession_per_orbit_1pn(a, e, C.GM_SUN, C.C_AU_DAY)
to_cy = C.ARCSEC_PER_RAD * C.JULIAN_CENTURY_DAYS / period
print(f"exact Schwarzschild: {exact * to_cy:.5f} arcsec/cy")
print(f"1PN formula:         {pn1 * to_cy:.5f} arcsec/cy")
print(f"difference:          {(exact - pn1) * to_cy:.2e} arcsec/cy  (relative {(exact - pn1) / pn1:.1e}; "
      f"the 2PN term is ~{7.5 * mu_sun / p:.1e}, below the numerical resolution)")

Where does 1PN fail? Shrink $c$ (equivalently, move the orbit inward in units of $\mu$) and
compare the exact advance with $6\pi\mu/p$. The ratio departs from 1 as $1 + 7.5\,\mu/p$ at
first, then faster; by $\mu/p \sim 0.03$ the advance is tens of degrees per orbit. And there is
a cliff: Mercury's Newtonian angular momentum is $L = \sqrt{GMp}$, i.e. $L/(\mu c) = \sqrt{p/\mu}$,
which drops below $\sqrt{12}$ once $p < 12\mu$ — at about $c/1900$ the well disappears and
Mercury, launched from its perihelion, falls into the Sun.

In [ ]:
print(f"{'c / c_real':>10s} {'mu/p':>9s} {'L/(mu c)':>9s} {'exact/1PN':>10s} {'1 + 7.5 mu/p':>13s} {'deg/orbit':>10s}")
for scale in (100.0, 300.0, 1000.0, 1500.0, 2000.0):
    c_s = C.C_AU_DAY / scale
    mu_s = C.GM_SUN / c_s ** 2
    L_over_muc = math.sqrt(p / mu_s)
    p1 = sch.precession_per_orbit_1pn(a, e, C.GM_SUN, c_s)
    try:
        ex = sch.precession_per_orbit_numeric(a, e, C.GM_SUN, c_s, n_orbits=20)
        print(f"{1 / scale:10.4f} {mu_s / p:9.2e} {L_over_muc:9.2f} {ex / p1:10.4f} {1 + 7.5 * mu_s / p:13.4f} {math.degrees(ex):10.2f}")
    except ValueError:
        print(f"{1 / scale:10.4f} {mu_s / p:9.2e} {L_over_muc:9.2f}   captured: no second perihelion (L < sqrt(12) mu c = 3.46)")

## 5. The strong-field zoo

Geometric units again. Each orbit is started at $r_0$ with a tangential proper velocity and
integrated with the Binet equation (`binet_orbit`); the returned $(\phi, r)$ is drawn in the
equatorial plane, in 3-D so the horizon ($r = 2$), photon sphere ($r = 3$) and ISCO ($r = 6$)
can be shown as rings.

* **precessing**: $L = 4.2$, started at $r_0 = 25$ (an "aphelion"): the perihelion advances by
  tens of degrees per orbit — the rosette of Lesson 04, but real;
* **zoom–whirl**: $L = 4.0$, energy a hair below the barrier top at $r = 4$: the particle
  zooms in, whirls several times near $r = 4$, zooms out again;
* **plunge**: same $L$, slightly more energy: over the barrier and into the horizon.

The energy is set through the initial *radial* velocity: $\mathcal E = \tfrac12 v_r^2 + V_{\rm eff}(r_0)$.

In [ ]:
def start(r0, L, energy, gm=1.0, c=1.0, inward=True):
    """2-D initial data at radius r0 with angular momentum L and total E_eff = energy."""
    v_phi = L / r0
    v_r2 = 2.0 * (energy - float(sch.effective_potential(r0, L, gm, c)))
    v_r = -math.sqrt(max(v_r2, 0.0)) if inward else math.sqrt(max(v_r2, 0.0))
    return np.array([r0, 0.0]), np.array([v_r, v_phi])


L_zw = 4.0
r_top = sch.circular_orbit_radii(L_zw, 1.0, 1.0)[1]  # unstable circular orbit (barrier top)
v_top = float(sch.effective_potential(r_top, L_zw, 1.0, 1.0))
cases = {
    "precessing (L = 4.2)": start(25.0, 4.2, float(sch.effective_potential(25.0, 4.2, 1.0, 1.0)), inward=False),
    "zoom-whirl (L = 4.0)": start(30.0, L_zw, v_top - 2e-5),
    "plunge (L = 4.0)": start(30.0, L_zw, v_top + 2e-3),
}
orbits = {}
for name, (r0, v0) in cases.items():
    phi, rr, t = sch.binet_orbit(r0, v0, 1.0, 1.0, n_orbits=6.0, n_per_orbit=3000)
    orbits[name] = (phi, rr, t)
    peri = sch.periapsis_angles(phi, rr)
    adv = np.degrees(np.diff(peri) - 2 * math.pi) if peri.size > 1 else []
    print(f"{name:22s} r_min = {rr.min():6.2f}, total turn = {abs(phi[-1] - phi[0]) / (2 * math.pi):5.2f} orbits, "
          f"advance per orbit = {np.array2string(np.asarray(adv), precision=1)} deg")

In [ ]:
def ring(radius, n=200):
    th = np.linspace(0, 2 * math.pi, n)
    return radius * np.cos(th), radius * np.sin(th), np.zeros(n)


fig = plt.figure(figsize=(13.0, 4.6), layout="constrained")
for k, (name, (phi, rr, t)) in enumerate(orbits.items()):
    ax = fig.add_subplot(1, 3, k + 1, projection="3d")
    ax.plot(rr * np.cos(phi), rr * np.sin(phi), np.zeros_like(rr), color=PALETTE[k], lw=0.9)
    for radius, lab, col in ((2.0, "horizon", "k"), (3.0, "photon sphere", "0.4"), (6.0, "ISCO", "0.7")):
        ax.plot(*ring(radius), color=col, lw=1.0, label=lab)
    lim = 32.0
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_zlim(-lim, lim); ax.set_box_aspect((1, 1, 1))
    ax.set_xlabel("x [GM/c²]"); ax.set_ylabel("y [GM/c²]"); ax.set_zlabel("z")
    ax.view_init(elev=55, azim=-60)
    ax.set_title(name)
    if k == 0:
        ax.legend(loc="upper left", fontsize=8)

## 6. Proper time and coordinate time

`geodesic_orbit` integrates the same geodesic in *proper* time $\tau$ (the clock carried by the
particle) and returns the coordinate time $t$ as well (the clock of a distant observer). Deep in
the well $dt/d\tau$ grows; near the horizon it diverges — the distant observer never sees the
plunging particle cross $r_s$.

In [ ]:
r0, v0 = cases["zoom-whirl (L = 4.0)"]
tau, t, rr, phi = sch.geodesic_orbit(r0, v0, 1.0, 1.0, tau_end=1500.0, n_samples=6000)
E, L = sch.conserved_quantities(r0, v0, 1.0, 1.0)
fig, axes = figure_grid(2, panel=(5.6, 3.6))
axes[0].plot(tau, rr, lw=0.9); axes[0].set_xlabel("proper time τ [GM/c³]"); axes[0].set_ylabel("r [GM/c²]")
axes[0].axhline(r_top, color="0.6", lw=0.8, ls="--"); axes[0].set_title("r(τ) of the zoom-whirl orbit")
axes[1].plot(tau, np.gradient(t, tau), lw=0.9); axes[1].set_xlabel("τ"); axes[1].set_ylabel("dt/dτ")
axes[1].set_title(f"time dilation along the orbit (E = {E:.5f} c²)")
for ax in axes:
    ax.grid(True, lw=0.3)
print(f"E = {E:.6f} c^2, L = {L:.3f} GM/c; far away dt/dtau -> E/c^2 = {E:.5f}; at r = 4: {E / (1 - 2 / 4):.4f}")

## 7. Summary

* $V_{\rm eff} = -GM/r + L^2/2r^2 - GML^2/c^2r^3$ contains the whole story: circular orbits,
  ISCO at $6\mu$, photon sphere at $3\mu$, capture.
* For Mercury the exact advance and 1PN agree to $10^{-7}$ relative; 1PN is off by 1 % at
  $\mu/p \sim 10^{-3}$ and meaningless by $\mu/p \sim 0.1$.
* The strong field produces precession of degrees per orbit, zoom–whirl orbits and plunges.

## Exercises

1. Show from $V_{\rm eff}$ that the ISCO energy is $E = \sqrt{8/9}\,c^2$: a particle spiralling
   down to the ISCO radiates 5.7 % of its rest mass (why accretion discs shine).
2. Starting from $r_0 = 10$, find by bisection the $L$ that separates capture from escape for a
   given energy. Compare with `circular_orbit_angular_momentum`.
3. ★ The zoom–whirl orbit whirls $n$ times for an energy deficit $\delta$ below the barrier top.
   Show numerically that $n \propto -\ln\delta$ and relate the coefficient to the Lyapunov
   exponent of the unstable circular orbit.
4. ★ Mercury's orbit in the exaggerated-$c$ runs of Lesson 04 used the 1PN *force*; redo
   §5 of that lesson with `binet_orbit` at $c/1000$ and quantify the 1PN error in the rosette.